In [1]:
# !pip install huggingface_hub ultralytics

# Imports

In [2]:
import yaml
import os

from ultralytics import YOLO
from huggingface_hub import snapshot_download
from pathlib import Path
import shutil
from IPython.display import Video

c:\Users\joelr\anaconda3\envs\computer_vision\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Carregar Base de Dados

A primeira tarefa para refinar um modelo é criar a base de dados.

In [3]:
local_repo_dir = snapshot_download(
    repo_id='johnatanvq/fruits-dataset',
    repo_type='dataset',
    allow_patterns=['fruitsData/**'] # Baixa somente esta pasta
)

print('Arquivo baixados em: ', local_repo_dir)

# 2) Mover/copiar para uma pasta final estilo ImageFolder (se quiser customizar o caminho)
src = Path(local_repo_dir) / 'fruitsData'
dst = Path('data/fruits') # pasta final onde você quer o ImageFolder

# Copiando arquivos para dst
if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(src, dst)

print('ImageFolder prontp em: ', dst)

Fetching 322 files: 100%|██████████| 322/322 [00:00<00:00, 5498.78it/s]


Arquivo baixados em:  C:\Users\joelr\.cache\huggingface\hub\datasets--johnatanvq--fruits-dataset\snapshots\2e9cf7d297327f8c1890ac1616be62c87d6fe5f5
ImageFolder prontp em:  data\fruits


In [4]:
def create_data_yaml(path_to_classes_txt, path_to_data_yaml):
    #Lê os arquivos "classes.txt"
    if not os.path.exists(path_to_classes_txt):
        print('classes.txt file not found! Please create a classes.txt leabelmap and move it to {path_to_classes_txt}')
        return
    with open(path_to_classes_txt, 'r') as f:
        classes = []
        for line in f.readlines():
            if len(line.strip()) == 0: continue
            classes.append((line.strip()))
    number_of_classes = len(classes)

    # Cria o dicionario a ser salvo
    data = {
        'path' : 'data/fruits',
        'train' : 'images',
        'val' : 'images',
        'nc' : number_of_classes,
        'names' : classes
    }

    #Escreve o arquivo YAML
    with open(path_to_data_yaml, 'w') as f:
        yaml.dump(data, f, sort_keys=False)
    print(f'Create config file at {path_to_data_yaml}')

    return

#Chama a função
create_data_yaml('data/fruits/classes.txt', 'yolo_train.yaml')

Create config file at yolo_train.yaml


In [5]:
str(Path().absolute())

'c:\\Users\\joelr\\repo\\nexvisual-Visao_Computacional'

In [ ]:
# Carrega o modelo pré-treinado
model = YOLO('yolo11n.pt')

# Treina o modelo utilizando as informações do arquivo YAML
# Definimos também a quantidade de épocas, o batch, e o tamanho das imagens.
results = model.train(data='./yolo_train.yaml', project=str(Path().absolute()), epochs=10, batch=2, imgsz=480)

Ultralytics 8.4.23  Python-3.12.12 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./yolo_train.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=480, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, p

In [ ]:
Video(r'videos\video.mov')

In [ ]:
model.predict(r'videos\video.mov', save=True, project=Path().absolute())